###10: Анонимизация данных

In [ ]:
import hashlib

OUTPUT_FILE = 'stations_anonymized.xlsx'

ID_COL      = 'Master Site'
DUP_ID_COL  = 'Номер сайта'
ADDR_COL    = 'Адрес'
LAT_COL     = 'Широта WGS84'
LON_COL     = 'Долгота WGS84'

df = df_main_with_weather.copy()

df.columns = df.columns.str.strip()

cols_to_drop = [col for col in [DUP_ID_COL, ADDR_COL] if col in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f" Удалены колонки: {cols_to_drop}")

SALT = "geo_project_2024"
def hash_station(code):
    if pd.isna(code):
        return np.nan
    return hashlib.sha256(f"{SALT}_{str(code).strip()}".encode()).hexdigest()[:12]

df[ID_COL] = df[ID_COL].apply(hash_station)
print(" Коды станций захэшированы")

valid_mask = df[LAT_COL].notna() & df[LON_COL].notna()
if valid_mask.any():
    coords = df.loc[valid_mask, [LON_COL, LAT_COL]].values.astype(float)

    SCALE      = 1.0          # Масштаб (1.0 = без изменения расстояний)
    ROTATION   = 37           # Угол поворота карты в градусах
    SHIFT      = (59.6, 30.2) # Сдвиг центра (примерно центр СПб/Ленобласти)
    NOISE_STD  = 0.004        # Гауссов шум (400 метров) для приватности

    theta = np.radians(ROTATION)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta), np.cos(theta)]])

    synth_coords = (coords * SCALE) @ R.T + np.array(SHIFT)
    synth_coords += np.random.default_rng(42).normal(0, NOISE_STD, synth_coords.shape)

    df.loc[valid_mask, LON_COL] = synth_coords[:, 0]
    df.loc[valid_mask, LAT_COL] = synth_coords[:, 1]
    print(" Координаты заменены на синтетические (геометрия сохранена)")

df.to_excel(OUTPUT_FILE, index=False)
print(f"\n Файл сохранён как: {OUTPUT_FILE}")

In [ ]:
df[LAT_COL].head()